# LAB 2 — Do texto e do número à evidência

**Disciplina** Tomada de Decisão Baseada em Dados para Gestores de Segurança Pública (CAES-PMERJ / FGV)
**Aula** 2 — Dados, IA e decisões com probabilidade (5 de novembro de 2026, presencial)
**Duração** Lab 1 das 10:00 às 11:45 (Partes 1 a 4) · Lab 2 das 14:45 às 16:30 (Partes 5 a 8)
**Entrega** Este notebook preenchido até a checagem final (Parte 8), antes da aula 3

> **Para quem é este lab:** gestores que decidem. Você não precisa programar. As células de cálculo vêm prontas; o grupo escreve entre aspas onde houver `TODO` e aperta Shift+Enter. O que importa é **interpretar** o resultado e **decidir**.

## Antes de começar

1. Importe no Databricks Free Edition: **Workspace → (seu usuário) → Import → File → `lab-2.ipynb`**.
2. Conecte ao compute *Serverless* e rode as células **na ordem**, de cima para baixo.
3. Se a internet do laboratório falhar, o notebook usa **dados de exemplo embutidos** (12 narrativas e 5 CISPs reais do ISP). O lab continua funcionando.

**Onde este lab está no arco.**

| Etapa | Neste lab |
| --- | --- |
| 2. Critérios | Julgamento com probabilidade e limiar, com `ai_decide` (Parte 6) |
| 3. Coleta e armazenamento | Narrativas e série do ISP na Bronze; anonimização antes de compartilhar (Partes 1 e 4) |
| 4. Processamento e visualização | Classificar, agrupar, comparar com a base certa e em taxa (Partes 2, 3 e 5) |
| 5. Decisão com critérios e risco | Aplicar o limiar e escolher o que levar para a aula 3 (Partes 6 e 7) |

**Fontes.** Narrativas: ALMEIDA, M. G. de; BOTEGA, L. C.; LIMA, P. M. de P. *Local Anonymization of Free-Text Robbery Police Reports in Brazilian Portuguese*, KDMiLe/SBC, 2026 — dataset CC BY 4.0, [repositório](https://github.com/murilogoes/bo-anonymizer-ptbr). Séries: [ISPDados](https://www.ispdados.rj.gov.br/), Instituto de Segurança Pública do RJ.

## Objetivos de aprendizagem

1. **Classificar** narrativas de boletins com regras e **auditar** o resultado à mão.
2. **Interpretar** grupos que a máquina encontra sozinha em textos.
3. **Medir** quanto dado pessoal um método de anonimização deixa vazar e decidir o que pode ser compartilhado.
4. **Comparar** uma série do ISP com a base certa e **normalizar** por população.
5. **Usar** um julgamento com probabilidade (`ai_decide`, no padrão System One visto na teoria) e aplicar um limiar de decisão.

## Parte 0 — O grupo e o recorte (2 min)

Use o mesmo recorte do lab 1.

In [0]:
# Parte 0 — o grupo e o recorte escolhido na aula 1.

GRUPO = 0          # TODO: número do grupo, de 1 a 8 (sem aspas)
CISP = 12          # TODO: número da CISP do grupo (sem aspas). 12 é só o exemplo da aula.
DELITO = "roubo_rua"  # TODO: "roubo_rua", "roubo_transeunte" ou "roubo_celular"

---

# Lab 1 (manhã) — NLP em boletins

## Parte 1 — Carregar as narrativas (10:00–10:15)

A célula abaixo baixa as 996 narrativas descaracterizadas do repositório público. Se não houver internet, usa uma amostra de 12 narrativas embutida. Nada aqui contém dado pessoal real: os nomes são fictícios.

In [0]:
%pip install openpyxl -q

In [0]:
# Célula pronta: carrega as narrativas (Bronze). Só execute.
import io
import json
import re
import unicodedata
import urllib.request

import numpy as np
import pandas as pd

def mostrar(df):
    try:
        display(df)
    except NameError:
        print(df.to_string())

URL_KDMILE = "https://github.com/murilogoes/bo-anonymizer-ptbr/raw/main/data/bos_anonimizacao_CAP_descaracterizado.xlsx"
AMOSTRA_EMBUTIDA = json.loads(r'''[
{
"Seq": 1,
"Texto Descaracterizado": "Pedro Paulo Da Silva foi preso em flagrante por roubo, após ter sido reconhecido pelas vítimas, Aloisio Freitas da Cunha e Patricia Hussein Brito, como o indivíduo que invadiu o estabelecimento comercial Prato Feliz, ameaçando-os com uma arma de fogo e subtraindo dinheiro, um celular da loja e o celular de Aloisio. A prisão foi convertida em preventiva, com base na gravidade do crime e no risco de reiteração delitiva.",
"Nome(s)": "Pedro Paulo Da Silva; Aloisio Freitas da Cunha; Patricia Hussein Brito; Aloisio",
"Localizações (s)": "Prato Feliz"
},
{
"Seq": 2,
"Texto Descaracterizado": "Presente nesta delegacia, Juliana Marta, informando que quando estava entrando em seu veículo que estava estacionado, foi abordada por um indivíduo dizendo que estava armado e que não era para ela gritar, senão iria atirar. O indivíduo levou uma corrente de ouro, 03 anéis e sua bolsa que continha seu celular, cartões de banco, RG e CNH. ",
"Nome(s)": "Juliana Marta",
"Localizações (s)": ""
},
{
"Seq": 3,
"Texto Descaracterizado": "Um cidadão foi abordado por dois indivíduos em uma moto Honda vermelha na Av. Dos Lagos, altura do número 367, no bairro Pedra Branca, por volta das 20:30h. O carona desceu da moto, anunciou o assalto e exigiu a mochila e os pertences da vítima, fugindo em seguida em direção desconhecida. O autor 1 permaneceu na moto e o autor 2, que desceu, era descrito como sem barba e sem óculos, vestindo camiseta do Corinthians. A placa da moto não foi identificada, mas a mesma estava amassada.",
"Nome(s)": "",
"Localizações (s)": "Av. Dos Lagos; Pedra Branca"
},
{
"Seq": 4,
"Texto Descaracterizado": "A vítima relatou que foi abordada por dois criminosos que a ameaçaram e a imobilizaram com uma \"gravata\", roubando seus pertences. A vítima não conseguiu identificar os autores do crime.",
"Nome(s)": "",
"Localizações (s)": ""
},
{
"Seq": 5,
"Texto Descaracterizado": "Comparecem os policiais militares Abreu e Vera, integrantes da VTR A12345, noticiando que durante deslocamento para atendimento de ocorrência,  avistaram uma motocicleta estacionada e com dois indivíduos nela embarcados. Em seguida, outra motocicleta, de modelo indeterminado, sendo pequeno porte e cor escura, com dois indivíduos, parou ao lado da primeira, um deles, desceu e foi ao encontro dos outros dois embarcados. A guarnição adentrou a via. Assim que os indivíduos avistaram a viatura, o que estava embarcado na moto de pequeno porte, fugiu na direção da Rua do Tanque. Os que estavam na Honda PCX, cor marrom, abandonaram a motocicleta no chão e empreenderam fuga, sendo que dois deles seguiram pelo telhado de uma casa vizinha e um deles foi alcançado. Próximo à Honda PCX foi localizado um boné e um aparelho de celular. Ao que indica, pertence a um dos indivíduos que fugiu. O abordado é o adolescente José Alfredo Pinto. Nada de ilícito foi encontrado, porém em busca nos sistemas policiais, a motocicleta é produto de roubo ocorrido nesta data às 18:30, nas proximidades do Shopping Botelho, na Avenida Ribeiro. Os PMs estavam com as câmeras ligadas, porém não acionadas. Foi utilizado algemas devido ao receio de fuga. A mãe do adolescente compareceu no local dos fatos. O adolescente, a motocicleta, o boné e o celular localizados foram conduzidos à esta delegacia e colocado à disposição da autoridade policial.  Presente nesta unidade, a vítima Maria Pereira da Silva nos relatou que estava a caminho do trabalho, por volta das 18:30, pela Travessa Sem Bar, altura do numeral 50, no município de Itaquaquecetuba, quando foi abordada por dois indivíduos numa moto, modelo não identificado. Eles interceptaram a sua moto após a lombada, ameaçaram com arma e ordenaram que ela descesse da sua motocicleta.  Não levaram capacete e nem celular, mas levaram o que estava no interior do baú. Ela seguiu até o posto de combustível próximo do local e acionou 190 para comunicar os fatos. Vítima compartilhou a notícia do roubo de sua moto nas redes sociais e foi avisada por uma amiga que viu uma moto parecida com a sua numa abordagem policial. Ela se dirigiu ao local e confirmou ser a sua motocicleta, sendo instruída pelos policiais a se dirigir a esta delegacia.  Na delegacia, foi realizado o Reconhecimento de Pessoa, mas a vítima não reconheceu o adolescente como sendo um dos autores do Roubo de sua motocicleta.  Indagado a respeito dos fatos, na presença de sua genitora, ora curadora, o adolescente  José Alfredo Pinto declarou que estava em casa quando decidiu ir visitar o amigo de nome Tubarão. No caminho, viu que uma viatura da PM estava abordando uma moto e os moleques jogaram esta moto no chão e saíram correndo. José se assustou, correu viu o portão aberto e entrou no quintal da casa de uma mulher. Na fuga, viu um outro moleque pulando o muro desta mesma casa. Os policiais entraram e pegaram José no quintal. Ordenaram que ele deitasse no chão, revistaram e perguntaram sobre a moto, mas ele desconhece a moto e os moleques que ali estavam. Informou, ainda que não faz uso de álcool e nem de entorpecentes. Ele não está estudando porque perdeu a vaga por excesso de faltas. A mãe informa que entrou com pedido de nova vaga, mas enquanto isso ele cuida dos irmãos para ela trabalhar. Seu genitor é ausente. Mãe informa que é a primeira vez que comparece em delegacia e ele nunca levou nada de diferente para dentro de casa. O que ele tem, ela comprou.  Ciente de tais fatos, após entrevista reservada com os envolvidos, nesta etapa urgente de cognição sumaríssima, este signatário formou sua decisão e convicção jurídica, entendendo não presentes elementos para se atribuir ao adolescente José a prática do roubo do veículo da vítima, restando imputação pelo ato infracional análogo ao delito prescrito no artigo 180 caput do Código Penal.  Não se tratando de ato infracional  praticado com violência ou grave ameaça, determino a liberação do adolescente infrator ao seu responsável legal, mediante a assinatura do termo de compromisso e responsabilidade, conforme preceitua o artigo 174 do Estatuto da Criança e do Adolescente, bem como a elaboração do presente boletim de ocorrência circunstanciado e seu posterior encaminhamento ao Ministério Público. ",
"Nome(s)": "Abreu; Vera; José Alfredo Pinto; Maria Pereira da Silva",
"Localizações (s)": "Rua do Tanque; Shopping Botelho; Avenida Ribeiro; Travessa Sem Bar"
},
{
"Seq": 6,
"Texto Descaracterizado": "Descrição ocorrência cidadão: Me acordou próximo ao Mercadinho do João aonde eu desci do ônibus e precisava me deslocar até o outro ponto  , me pediu dinheiro , informei que não tinha  , começou a ficar apreensivo.  Agressivo em palavras , começou a tentar puxa minha mochila.  Puxou e saiu correndo ,  arranhou meu braço  Histórico do boletim: A redação do presente Boletim Eletrônico de Ocorrência foi elaborada pelo Declarante e as informações nele constantes são de sua exclusiva responsabilidade.     Este B.E.O. não substitui os documentos originais.    IMPORTANTE:   - Caso RG emitido no Estado de São Paulo tenha sido roubado e listado neste B.E.O., ficará bloqueado em caráter irreversível e nova via deverá ser providenciada;   - Mesmo que os documentos sejam recuperados, não haverá possibilidade de cancelamento deste B.E.O..  BO digitado por Maria Pereira Silva, equipe Prodesp.   - Autor 1 – Desconhecido, Barba: Barba cheia ",
"Nome(s)": "Maria Pereira Silva",
"Localizações (s)": "Mercadinho do João"
},
{
"Seq": 7,
"Texto Descaracterizado": "Descrição ocorrência cidadão: Estava Descendo a Rua para chegar em casa , fui abordado por dois indivídos em uma moto de características preta os dois estavam de moletom preto capacete preto um deles com detalhe vermelho no capacete , apontaram a arma na minha cabeça e pegaram o celular pediram carteira mas não tinha somente o celular , a última localização do aparelho estava constando entre a Rua do Tanque, Avenida Ribeiro, 2000 Histórico do boletim: A redação do presente Boletim Eletrônico de Ocorrência foi elaborada pelo Declarante e as informações nele constantes são de sua exclusiva responsabilidade.     IMPORTANTE:   Mesmo que o aparelho seja recuperado, não haverá possibilidade de cancelamento deste B.E.O..    B.O Digitado por João Andrade de Souza, Equipe Prodesp. - Autor 1 – Desconhecido, Barba: Sem barba  - Autor 2 – Desconhecido, Barba: Sem barba   VEÍCULO: Não Especificado Pessoa Relacionada: Autor 1 – Desconhecido Placa: Não informado Tipo de Veículo: Moto Marca/Modelo: Não informado Cores: Preta Estado (UF): SP Cidade: Não informado Proprietário do Veículo: Não informado Características adicionais do veículo úteis para a investigação: Moto preta aparentava ser uma 160 Com a placa meio dobrada . ",
"Nome(s)": "João Andrade de Souza",
"Localizações (s)": "Rua do Tanque; Avenida Ribeiro, 2000"
},
{
"Seq": 8,
"Texto Descaracterizado": "Um cidadão foi abordado por dois indivíduos em uma moto preta, na Rua do Tanque, enquanto caminhava para casa. Os indivíduos, que usavam moletom preto e capacetes pretos (um com detalhe vermelho), apontaram uma arma para a vítima e roubaram seu celular. A última localização do aparelho foi registrada entre a Rua do Tanque e a Avenida Ribeiro.",
"Nome(s)": "",
"Localizações (s)": "Rua do Tanque; Avenida Ribeiro"
},
{
"Seq": 10,
"Texto Descaracterizado": "Descrição ocorrência cidadão: 5 ou 4 caras montado em 3 motos vieram pelo lado do parceiro e anunciou o assalto só tinha um armado com um 38 mais ou menos só lembro que era uma arma com os negócios que gira as balas uma moto era branca ???  ... ???     ... CONSIDERAÇÕES DO DELEGADO DE POLÍCIA: a narrativa foi realizada pela própria vítima, utilizando-se da plataforma do Boletim de Ocorrência Eletrônico e, por ora, com a análise dos dados firmados pelo usuário dos serviços, conclui-se que o caso tem feições do crime acima especificado. .  Caso a vítima tenha conhecimento de informações que possam elucidar a determinação de autoria, fica desde já orientada a comparecer no Setor de Investigações da Delegacia de Polícia do local dos fatos, juntamente com o presente Boletim de Ocorrência, para compartilhamento de informações.   Registro a ser levado para deliberação do Exmo. Sr. Dr. Delegado de Polícia titular da unidade do local dos fatos. Nada mais.  ",
"Nome(s)": "",
"Localizações (s)": ""
},
{
"Seq": 11,
"Texto Descaracterizado": "Comparecem nesta Delegacia de Polícia os policiais militares acima qualificados narrando os seguintes fatos - que na data de 14/04/2005 quatro indivíduos tentaram roubar o estabelecimento comercial “Farmácia do Souza” situado no endereço, Rua do Tanque, numeral 100, bairro Poiares. O gerente do citado estabelecimento ao perceber a atitude suspeita dos indivíduos, solicitou para que um outro funcionário, acionasse o alarme do local. Diante do fato, os indivíduos se evadiram do interior do estabelecimento, e ainda tentaram roubar um cliente que estava no estacionamento. Porém não obtiveram êxito.  Seguiram então sentido região sul desta cidade, momento em que abordaram a pessoa de João Andrade de Souza que naquele momento estava se dirigindo para o trabalho, e fazendo menção de que estavam armados, solicitaram o celular dele. Via COPOM, a equipe policial foi notificada sobre a tentativa de roubo na “Farmácia do Souza”. Prontamente se dirigiram ao local onde o gerente narrou o ocorrido acima, e informou que os indivíduos teriam fugido no sentido sul desta cidade. Ato contínuo em busca dos indivíduos, a equipe policial avistou um indivíduo, no caso a vítima, solicitando auxílio. Vítima os alertou de que havia sido roubado, e apontou a direção que os indivíduos tomaram rumo. Em abordagem pessoal, nada de ilícito foi encontrado com os indivíduos, porém um deles, José Alfredo Pinto, estava com o objeto utilizado para efetuar o roubo do aparelho celular, no caso uma tesoura. O aparelho celular no momento estava em posse do indiciado.  Ante ao exposto, a equipe policial conduziu a vítima e os quatro indivíduos até este distrito policial para a apreciação da autoridade policial plantonista. Foi necessário o uso de algemas no indiciado e nos adolescentes infratores tanto para a segurança deles quanto da equipe policial. Os indivíduos não se encontram lesionados. Expediu-se requisição de exame de corpo de delito cautelar, visando atestar a integridade física dos indivíduos, sendo determinada a fotografação deles de corpo inteiro.  I - DAS OITIVAS 1. Depoimento da Vítima A vítima João Andrade de Souza declarou que foi abordada pouco antes das 3h da madrugada quando se dirigia ao trabalho de bicicleta. Relatou que quatro indivíduos em bicicletas o imprensaram contra um ponto de ônibus, fizeram menção de estarem com arma de fogo escondida sob uma blusa de frio e apontaram uma tesoura para si. Os indivíduos o revistaram e subtraíram seu aparelho celular, ordenando que não olhasse para trás ao sair. Após se afastar, encontrou uma viatura policial, relatou o ocorrido e, posteriormente, reconheceu o celular recuperado pelos policiais como sendo de sua propriedade. A vítima afirmou com convicção reconhecer os quatro indivíduos apresentados como os autores do roubo.  2. Depoimento dos Policiais Militares Condutores O PM Abreu relatou que a equipe policial foi acionada via rede rádio para averiguar uma tentativa de roubo à Farmácia do Souza. Durante o deslocamento, foram solicitados por um transeunte (vítima João) que informou ter sido roubado por quatro indivíduos, com características compatíveis com os autores da tentativa de roubo à farmácia. A vítima informou a direção de fuga dos suspeitos, e a equipe policial os localizou embaixo da ponte da Besta, no bairro Porto Novo . Na abordagem, foi encontrado o celular roubado com o indivíduo maior de idade. O policial confirmou o uso de algemas, mas negou resistência na abordagem.  O PM Costa corroborou o depoimento do seu parceiro, acrescentando que durante a revista pessoal, o aparelho celular foi localizado com o maior de idade, e no último indivíduo (José Alfredo Pinto) foi encontrada uma tesoura. Informou também que a vítima relatou que os autores simularam estar armados. Confirmou o uso de algemas justificado pelo receio de fuga e pela superioridade numérica dos abordados.  3. Interrogatório do Indiciado Guilherme Pereira, 18 anos, ao ser interrogado, exerceu seu direito constitucional de permanecer em silêncio. Ao final do interrogatório, quando questionado sobre as condições da abordagem policial, alegou ter recebido \"um soco na cabeça\" e que as algemas foram colocadas com muita força.  4. Depoimento dos Adolescentes Infratores Marcos dos Santos, 15 anos, optou por permanecer em silêncio, exercendo seu direito constitucional de não produzir provas contra si mesmo.  Pedro Minotauro, 15 anos, inicialmente optou por prestar declarações, confirmando que esteve na farmácia com os demais, todos encapuzados. Alegou que não fizeram nada na farmácia e que teriam ido embora. Quando questionado sobre o encontro com a vítima, apresentou contradições em suas declarações, inicialmente confirmando e depois negando o envolvimento no roubo do celular.  José Alfredo Pinto, 16 anos, optou por permanecer em silêncio, exercendo seu direito constitucional.  II - DA MATERIALIDADE E AUTORIA A materialidade delitiva encontra-se demonstrada pelo Auto de Exibição e Apreensão que documenta a apreensão do aparelho celular da vítima em poder do indiciado Guilherme Pereira, bem como da tesoura utilizada como instrumento para intimidação durante o roubo, encontrada com o adolescente José Alfredo Pinto. Consta também a Requisição de Exame Pericial dos objetos apreendidos.  III - DA TIPIFICAÇÃO Diante do conjunto probatório colhido, verifico que a conduta do indiciado Guilherme Pereira se amolda, em tese, aos tipos penais previstos no artigo 157, §2º, inciso II do Código Penal (roubo qualificado pelo concurso de pessoas) e artigo 244-B do Estatuto da Criança e do Adolescente (corrupção de menores), em concurso material (art. 69 do CP).  Quanto aos adolescentes José Alfredo Pinto, Marcos dos Santos e Pedro Minotauro, suas condutas caracterizam, em tese, ato infracional análogo ao crime previsto no artigo 157, §2º, inciso II do Código Penal.  iV - DA HOMOLOGAÇÃO DO FLAGRANTE Considerando que a prisão em flagrante atende aos requisitos legais estabelecidos nos artigos 301 e 302, incisos I e II do Código de Processo Penal, não apresentando quaisquer vícios formais ou materiais, HOMOLOGO o presente auto de prisão em flagrante.  V - DELIBERAÇÕES Ante o exposto, DETERMINO:  A LAVRATURA DO AUTO DE PRISÃO EM FLAGRANTE do indiciado Guilherme Pereira, nos termos do Código de Processo Penal;  A LAVRATURA DO AUTO DE APREENSÃO dos adolescentes José Alfredo Pinto, Marcos dos Santos e Pedro Minotauro, nos termos do Estatuto da Criança e do Adolescente;  A expedição de nota de culpa ao autuado;  A expedição de ofício à autoridade judiciária competente, comunicando a prisão;  A expedição de ofício ao Ministério Público, para conhecimento;  A instauração do Inquérito Policial para apuração completa dos fatos;  A remessa dos adolescentes infratores ao Juízo da Infância e Juventude;  O encaminhamento do indiciado maior de idade ao sistema prisional.",
"Nome(s)": "João Andrade de Souza; José Alfredo Pinto; João; Guilherme Pereira; Marcos dos Santos; Pedro Minotauro; Abreu; Costa",
"Localizações (s)": "Farmácia do Souza; Rua do Tanque, numeral 100; ponte da Besta; bairro Poiares; bairro Porto Novo"
},
{
"Seq": 12,
"Texto Descaracterizado": "Neste dia e horário, na sede da Central de Polícia Judiciária de Praia Grande, comparecem os Policiais Militares acima identificados narrando que estavam em patrulhamento de rotina, quando foram informados pela VTR A12345, encarregado CB PM Abreu, que um indivíduo havia subtraído pertences e dinheiro das vítimas do estabelecimento comercial de nome Restaurante Coma Bem, situado Rua do Tanque, nº. 100, bairro Boqueirão, indicando as características do autor. Informou ainda que o rastreio do celular da vítima estava no bairro Sítio do Campo, em deslocamento pelas ruas daquela localidade. Em patrulhamento com vistas, o indivíduo foi localizado de bicicleta e abordado na Avenida Ribeiro, altura do nº. 2.000, bairro Sítio do Campo, identificado como João Andrade de Souza. Segundos antes da abordagem, avistaram o autor dispensando ao chão algo, o qual foi recuperado e se tratava de um objeto metálico simulando uma arma de fogo. Após busca pessoal, na mão do autor foram localizados dois celulares e moedas, envoltos em uma blusa. Diante dos fatos, foi conduzido a esta Delegacia de Polícia para as providências cabíveis, sendo necessário o uso de algemas, diante de fundado receio de fuga. A bicicleta foi apreendida administrativamente pela Polícia Militar.  A ofendida Maria Pereira Silva afirma que trabalha no estabelecimento comercial Restaurante Coma Bem, na função de auxiliar de cozinha, e estava limpando a cozinha para fechar e José estava na frente recebendo os pedidos. Em dado momento, um cidadão adentrou na cozinha, com uma arma nas costas do José, exigindo que a declarante entregasse dinheiro e seu celular. Havia um celular carregando no balcão da cozinha e o autor o puxou, apoderando-se do bem, que é de propriedade de José. A declarante disse que era funcionária e não tinha dinheiro. O indivíduo voltou para a parte da frente e começou a revirar o caixa. Ao fim, o indivíduo se evadiu, tomando rumo ignorado. A genitora de José desceu as escadas e acionou a Polícia. Atesta que não teve prejuízo financeiro. Após reconhecimento pessoal, reconhece o agora identificado João Andrade de Souza como autor do delito, com 100% de certeza.   O ofendido José Alfredo Pinto, na presença de sua genitora, afirma que possui quatorze anos de idade e deseja revelar espontaneamente o ocorrido a este Delegado de Polícia. No horário dos fatos, estava na parte da frente do estabelecimento comercial Restaurante Coma Bem, recebendo os pedidos online de clientes. A porta da frente estava trancada, pois é delivery. Em dado momento, um homem tentou abrir a porta e perguntou se a loja estava aberta. O declarante disse que sim e foi lá abrir, pois pensou que se tratava de um cliente. O indivíduo entrou e, quanto o declarante se virou para o balcão, foi abordado com um objeto parecido com arma, que disse: “se afasta; se afasta”, gesticulando, então o declarante entrou na cozinha. O homem passou a revirar o caixa e se apoderou das notas e moedas. O indivíduo visualizou um celular que estava carregando em cima da bancada e se apoderou, puxando, celular este que é da loja. Pediu que o declarante saísse da cozinha e o confrontou se o que tinha no caixa era tudo o que tinha no comércio. O declarante afirmou que sim e o autor olhou novamente o balcão e se apoderou do seu celular que estava em cima do balcão do caixa, ao lado do monitor. Ao fim, o indivíduo se evadiu. Começou a gritar por sua mãe que estava na parte de cima do estabelecimento, a qual desceu e se inteirou do ocorrido, pois nada presenciou. Sua genitora acionou a Polícia Militar, passando a localização do telefone. Após reconhecimento pessoal, reconhece o agora identificado João Andrade de Souza como autor do delito, com 100% de certeza. Seu celular se trata de um Samsung Galaxy M55, avaliado em R$ 1.500,00 (mil e quinhentos reais); o da loja se trata de um Multilaser F Pro 2, avaliado em R$ 400,00 (quatrocentos reais), além da quantia de R$ 58,31 (cinqüenta e oito reais e trinta e um centavos) em dinheiro do caixa.  João Andrade de Souza, antes de ser ouvido por esta Autoridade Policial foi cientificado de seus direitos constitucionais, especialmente o de permanecer em silêncio durante o interrogatório policial e falar somente em juízo. Declara que irá se manifestar apenas em juízo. Não indicou o número de ninguém para ser informado de sua prisão  Examinadas as versões e demais elementos amealhados, o Delegado de Policia exarou sua decisão e convicção jurídica (Lei Federal n°12.830/2013, art.2°, parágrafo 6°): os elementos apresentados, em especial a firme narrativa dos ofendidos e o reconhecimento pessoal, bem como a forma que se deu a abordagem, com o autor ainda em posse da res furtiva, são suficientes para apontar a prática do crime tipificado no artigo 157, caput, do Código Penal.  Por tal razão, foi DECRETADA, neste instante, a PRISÃO EM FLAGRANTE de João Andrade de Souza, determinando-se o prosseguimento do procedimento nos seus ulteriores termos.  A Autoridade Policial, na etapa de cognição sumaríssima, reputou configurado o estado flagrancial, já que o autor tinha acabado de cometer a infração penal (CPP, art. 302, II).   Considerando a pena máxima cominada em abstrato suplanta quatro anos, deixo de arbitrar fiança, permanecendo o indiciado custodiado. Determino ainda o FORMAL INDICIAMENTO do agente, com entrega da correspondente nota de culpa.   Autorizo a entrega da res furtiva ao legítimo proprietário, presentes a este plantão policial. Determino ainda que os vídeos das câmeras de segurança do comércio sejam disponibilizados via Portal de Vídeos. Determino a apreensão do objeto utilizado como “arma” para ameaçar as vítimas.   Comunique-se a Autoridade Judicial competente, com promoção das demais comunicações e cerimônias legais pertinentes. Juntem-se, ademais, oitivas e documentação pertinente, integrantes deste. Instaure-se Inquérito Policial.   DA REPRESENTAÇÃO PARA CONVERSÃO DA PRISÃO EM FLAGRANTE EM PRISÃO PREVENTIVA  Conforme previsão do art. 282, incisos I e II do CPP, a aplicação das medidas cautelares se dá de acordo com os Princípios da Necessidade e da Adequação. No caso em comento, a segregação cautelar se mostra como a medida mais necessária e adequada.  Diante da gravidade concreta do crime, bem como das circunstâncias do autor e do risco de ineficácia de outras medidas cautelares diversas da prisão, REPRESENTA-SE pela CONVERSÃO DA PRISÃO EM FLAGRANTE EM PRISÃO PREVENTIVA do indiciado João Andrade de Souza, nos termos dos arts. 282 § 6º, 311, 312, e 313 inc. I, todos do Código de Processo Penal, a fim de garantir da ordem pública e para assegurar a aplicação da lei penal. ",
"Nome(s)": "Abreu; João Andrade de Souza; Maria Pereira Silva; José; José Alfredo Pinto",
"Localizações (s)": "Restaurante Coma Bem; Rua do Tanque, nº. 100; bairro Boqueirão; Avenida Ribeiro, altura do nº. 2.000; bairro Sítio do Campo"
},
{
"Seq": 13,
"Texto Descaracterizado": "O declarante, após pegar o trem errado, pediu informações a um morador de rua, que lhe roubou o celular. Outro morador interveio, recuperando o aparelho, mas a capinha e os documentos que estavam dentro foram perdidos. O Boletim de Ocorrência foi registrado e o RG do declarante, caso tenha sido emitido em São Paulo, ficará bloqueado.",
"Nome(s)": "",
"Localizações (s)": ""
}
]''')

try:
    with urllib.request.urlopen(URL_KDMILE, timeout=30) as r:
        narrativas = pd.read_excel(io.BytesIO(r.read()))
    FONTE_NARRATIVAS = "repositório público (996 narrativas)"
except Exception as erro:
    narrativas = pd.DataFrame(AMOSTRA_EMBUTIDA)
    FONTE_NARRATIVAS = f"amostra embutida (12 narrativas) — download falhou: {type(erro).__name__}"

narrativas = narrativas.rename(columns={"Texto Descaracterizado": "texto"})
narrativas["texto"] = narrativas["texto"].fillna("").astype(str)
print(f"Fonte: {FONTE_NARRATIVAS}")
print(f"{len(narrativas)} narrativas carregadas. Exemplo:\n")
print(narrativas["texto"].iloc[1][:500])

## Parte 2 — Classificar por regras e auditar (10:15–10:45)

**Atividade 2.1 (10 min).** A célula usa um dicionário de palavras para marcar **o que foi levado** e **o meio empregado**. É o jeito mais simples de NLP: transparente e fácil de corrigir. O grupo pode acrescentar palavras ao dicionário (opcional) e rodar de novo.

In [0]:
# Atividade 2.1 — dicionário de regras. Opcional: acrescente palavras (sem acento, minúsculas).

OBJETO = {
    "Celular": ["celular", "telefone movel", "smartphone", "iphone"],
    "Veículo (carro ou moto)": ["veiculo", "carro", "automovel", "moto", "motocicleta"],
    "Estabelecimento comercial": ["loja", "estabelecimento", "comercio", "mercado", "farmacia", "padaria"],
    "Residência": ["residencia", "casa da vitima", "apartamento"],
    "Transporte coletivo": ["onibus", "coletivo"],
    "Bicicleta": ["bicicleta"],
}
MEIO = {
    "Arma de fogo": ["arma de fogo", "revolver", "pistola", "armado"],
    "Arma branca": ["faca", "canivete", "arma branca", "facao"],
    "Simulacro": ["simulacro", "arma de brinquedo"],
}

In [0]:
# Célula pronta: aplica as regras e conta. Só execute.

def sem_acento(t):
    return unicodedata.normalize("NFKD", t.lower()).encode("ascii", "ignore").decode()

def marcar(texto, dicionario):
    t = sem_acento(texto)
    return [rotulo for rotulo, palavras in dicionario.items() if any(p in t for p in palavras)]

narrativas["objeto"] = narrativas["texto"].map(lambda t: marcar(t, OBJETO))
narrativas["meio"] = narrativas["texto"].map(lambda t: (marcar(t, MEIO) or ["Sem arma descrita"])[0])

contagem = (narrativas["objeto"].explode().fillna("Nenhum objeto identificado")
            .value_counts().rename_axis("O que foi levado").reset_index(name="Narrativas"))
mostrar(contagem)
mostrar(narrativas["meio"].value_counts().rename_axis("Meio empregado").reset_index(name="Narrativas"))

**Atividade 2.2 (15 min) — auditoria.** Toda classificação automática precisa ser conferida. A célula sorteia 10 narrativas (o sorteio muda com o número do grupo). Leiam cada uma e digam se as marcas estão **certas**, **erradas** ou **incompletas**.

In [0]:
# Célula pronta: sorteia 10 narrativas para auditar. Só execute.
auditar = narrativas.sample(min(10, len(narrativas)), random_state=GRUPO or 1)
for i, (_, linha) in enumerate(auditar.iterrows(), start=1):
    print(f"[{i}] objeto={linha['objeto']} | meio={linha['meio']}")
    print("    " + linha["texto"][:350].replace("\n", " ") + ("..." if len(linha["texto"]) > 350 else ""))
    print()

In [0]:
# Atividade 2.2 — uma avaliação por narrativa sorteada: "certo", "errado" ou "incompleto".

AUDITORIA = ["TODO"] * 10   # TODO: ex. ["certo", "certo", "incompleto", "errado", ...]

validas = [a for a in AUDITORIA if a in ("certo", "errado", "incompleto")]
if validas:
    print(f"Acerto das regras na auditoria: {validas.count('certo')}/{len(validas)} "
          f"({validas.count('certo') / len(validas):.0%})")
    print("Para comparar: modelos BERT chegaram a 89%–90% de acurácia em 10 classes de crime (Alves et al., 2024).")
else:
    print("Preencha a auditoria e rode de novo.")

## Parte 3 — O que a máquina encontra sozinha (10:45–11:05)

Agora sem regras: a célula transforma cada narrativa em números (TF-IDF, o peso de cada palavra) e junta as parecidas em 6 grupos (k-means). Ninguém disse à máquina o que procurar. O trabalho do gestor é **dar nome** a cada grupo e dizer se ele serve para alguma decisão.

In [0]:
# Célula pronta: agrupa as narrativas em até 6 grupos e mostra as palavras de cada um. Só execute.
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

PALAVRAS_VAZIAS = """a o e é de da do das dos em no na nos nas um uma uns umas que se por para com sem ao aos à às
pelo pela pelos pelas seu sua seus suas ele ela eles elas foi era estava ser ter tinha como mais mas ou já não sim lhe
lhes quando onde após sobre entre até também isso este esta esse essa vítima vitima indivíduo individuo indivíduos
individuos declarante relata informa informou momento local dia data hora horas cerca""".split()

k = min(6, max(2, len(narrativas) // 3))
vetor = TfidfVectorizer(stop_words=PALAVRAS_VAZIAS, max_df=0.5, min_df=2 if len(narrativas) < 100 else 5,
                        token_pattern=r"(?u)\b[^\W\d_]{3,}\b")
X = vetor.fit_transform(narrativas["texto"])
modelo = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
narrativas["grupo_texto"] = modelo.labels_
termos = np.array(vetor.get_feature_names_out())
resumo = pd.DataFrame({
    "Grupo": range(k),
    "Narrativas": [int((modelo.labels_ == g).sum()) for g in range(k)],
    "Palavras mais marcantes": [", ".join(termos[np.argsort(modelo.cluster_centers_[g])[::-1][:8]]) for g in range(k)],
})
mostrar(resumo)

In [0]:
# Atividade 3.1 — deem um nome a cada grupo e digam se ele ajuda alguma decisão.

NOMES_DOS_GRUPOS = {g: "TODO" for g in range(k)}   # TODO: ex. {0: "roubo por dupla em moto", 1: "registro eletrônico", ...}
GRUPO_MAIS_UTIL = "TODO"                          # TODO: qual grupo ajuda uma decisão do comando, e qual decisão?

## Parte 4 — Anonimizar e medir o que vaza (11:05–11:35)

O histórico do BO não é público porque tem nomes, locais, placas e documentos (LAI, art. 31; LGPD). Para compartilhar texto, é preciso anonimizar. A célula testa dois níveis de um anonimizador simples e mede, contra o padrão-ouro do corpus, **quanto de cada tipo de dado foi removido** (recall de segurança).

- **Nível 1:** troca placas, telefones, números de documento e sequências de duas ou mais palavras com inicial maiúscula.
- **Nível 2:** também troca palavras isoladas com inicial maiúscula fora do início de frase. Remove mais, mas apaga mais texto útil.

**Atividade 4.1.** A Corregedora escolhe o nível e o grupo decide com quem o texto anonimizado pode ser compartilhado.

In [0]:
# Atividade 4.1 — escolha o nível de anonimização a testar.
NIVEL = 1   # TODO: 1 ou 2

In [0]:
# Célula pronta: anonimiza e mede o recall contra o padrão-ouro. Só execute.
MAI, MIN = "A-ZÁÉÍÓÚÂÊÔÃÕÇ", "a-záéíóúâêôãõç"
PADROES = {
    "placa": re.compile(r"\b[A-Z]{3}-?\d[A-Z0-9]\d{2}\b"),
    "telefone": re.compile(r"\(?\d{2}\)?\s?9?\d{4}-?\d{4}"),
    "documento": re.compile(r"\b\d{1,3}(\.\d{3}){2}-?[\dXx]{1,2}\b|\b\d{9,11}\b"),
    "nome_1": re.compile(rf"\b[{MAI}][{MIN}]+(?:\s+(?:d[aeo]s?\s+)?[{MAI}][{MIN}]+)+"),
    "nome_2": re.compile(rf"(?<![.!?:]\s)(?<!^)\b[{MAI}][{MIN}]+(?:\s+(?:d[aeo]s?\s+)?[{MAI}][{MIN}]+)*"),
}

def anonimizar(texto, nivel):
    t = PADROES["placa"].sub("[PLACA]", texto)
    t = PADROES["telefone"].sub("[TELEFONE]", t)
    t = PADROES["documento"].sub("[DOCUMENTO]", t)
    return PADROES["nome_1" if nivel == 1 else "nome_2"].sub("[NOME]", t)

GABARITO = {"Nome(s)": "Nome", "Localizações (s)": "Localização", "Documento(s)": "Documento",
            "Emplacamento (s)": "Placa", "Telefone (s)": "Telefone"}

def medir(nivel):
    anon = narrativas["texto"].map(lambda t: anonimizar(t, nivel))
    linhas = []
    for coluna, rotulo in GABARITO.items():
        if coluna not in narrativas:
            continue
        total = vazou = 0
        for gabarito, texto in zip(narrativas[coluna], anon):
            if pd.isna(gabarito) or not str(gabarito).strip():
                continue
            for entidade in str(gabarito).split(";"):
                entidade = entidade.strip()
                if len(entidade) >= 2:
                    total += 1
                    vazou += entidade in texto
        if total:
            linhas.append({"Tipo": rotulo, "Entidades no gabarito": total, "Vazaram": vazou,
                           "Removidas (recall)": f"{1 - vazou / total:.0%}"})
    mascaradas = int(anon.str.count(r"\[NOME\]|\[PLACA\]|\[TELEFONE\]|\[DOCUMENTO\]").sum())
    return anon, pd.DataFrame(linhas), mascaradas

narrativas["anonimizado"], RESULTADO_ANON, MASCARADAS = medir(NIVEL)
_, _, mascaradas_outro = medir(2 if NIVEL == 1 else 1)
print(f"Nível {NIVEL}: {MASCARADAS} trechos mascarados (no outro nível seriam {mascaradas_outro}).")
mostrar(RESULTADO_ANON)
print("\nAntes:  " + narrativas["texto"].iloc[0][:300])
print("Depois: " + narrativas["anonimizado"].iloc[0][:300])

In [0]:
# Atividade 4.2 — a decisão da Corregedoria.

DECISAO_ANONIMIZACAO = {
    "nivel_escolhido": "TODO",          # "1" ou "2"
    "pode_compartilhar_com": "TODO",    # ex. "só a Inteligência", "batalhões", "ninguém fora da força"
    "justificativa": "TODO",            # uma frase: o que ainda vaza e por que isso é aceitável (ou não)
}

**Para a plenária (11:35):** se o nível 2 remove 9 de cada 10 nomes, os que sobram justificam publicar? O que isso diz sobre o histórico do BO não ser público?

---

# Lab 2 (tarde) — Checagem de notícias e dashboard

## Parte 5 — A série do ISP da CISP do grupo (14:45–15:25)

A célula baixa as séries públicas do ISP (ocorrências por CISP e mês, e população por CISP). Se falhar, usa dados reais embutidos de 5 CISPs (1, 5, 12, 33 e 64), de out/2024 a set/2026.

> **Atenção:** no ISP, os três meses mais recentes são **preliminares** (fase 2) e podem mudar. A população por CISP é a última estimativa do ISP (jul/2022).

In [0]:
# Célula pronta: carrega as séries do ISP (Bronze → Silver). Só execute.
URL_CISP = "https://www.ispdados.rj.gov.br/Arquivos/BaseDPEvolucaoMensalCisp.csv"
URL_POP = "https://www.ispdados.rj.gov.br/Arquivos/PopulacaoEvolucaoMensalCisp.csv"
PLANO_B = json.loads('''{"1": {"roubo_rua": [71, 44, 52, 77, 168, 219, 96, 99, 85, 74, 48, 56, 92, 79, 90, 114, 149, 75, 56, 71, 63, 66, 72, 55], "roubo_transeunte": [49, 20, 30, 47, 115, 134, 55, 69, 50, 39, 25, 34, 58, 32, 50, 66, 93, 42, 35, 48, 38, 35, 45, 30], "roubo_celular": [19, 18, 16, 23, 51, 81, 33, 23, 26, 32, 20, 20, 33, 44, 39, 44, 53, 32, 21, 20, 25, 29, 25, 25]}, "5": {"roubo_rua": [111, 117, 181, 196, 200, 425, 137, 158, 124, 121, 89, 96, 126, 110, 188, 240, 388, 204, 161, 169, 140, 151, 103, 136], "roubo_transeunte": [63, 76, 99, 117, 110, 262, 85, 85, 62, 58, 46, 52, 66, 56, 84, 127, 202, 99, 76, 85, 74, 63, 53, 53], "roubo_celular": [46, 38, 79, 71, 86, 157, 52, 71, 61, 62, 41, 43, 53, 50, 103, 111, 182, 104, 85, 82, 64, 87, 49, 83]}, "12": {"roubo_rua": [26, 25, 30, 57, 17, 32, 20, 64, 20, 28, 34, 22, 22, 25, 24, 36, 32, 25, 25, 70, 26, 20, 21, 20], "roubo_transeunte": [18, 17, 14, 36, 9, 21, 12, 24, 12, 16, 12, 11, 15, 13, 10, 19, 16, 16, 14, 35, 8, 10, 14, 10], "roubo_celular": [8, 7, 13, 18, 7, 11, 7, 39, 6, 12, 19, 11, 4, 11, 12, 17, 15, 9, 9, 34, 18, 10, 5, 9]}, "33": {"roubo_rua": [80, 74, 75, 92, 105, 88, 78, 54, 73, 85, 67, 59, 57, 32, 34, 60, 40, 56, 37, 52, 37, 49, 33, 38], "roubo_transeunte": [14, 22, 14, 22, 8, 15, 21, 18, 21, 27, 15, 22, 16, 12, 9, 25, 15, 24, 11, 11, 8, 13, 10, 10], "roubo_celular": [62, 50, 50, 56, 84, 69, 52, 35, 44, 49, 46, 37, 39, 18, 23, 34, 25, 30, 25, 32, 26, 33, 20, 27]}, "64": {"roubo_rua": [159, 159, 131, 179, 145, 139, 152, 146, 227, 224, 214, 133, 153, 91, 150, 127, 155, 146, 124, 192, 174, 148, 188, 182], "roubo_transeunte": [88, 86, 69, 63, 82, 80, 73, 82, 77, 98, 68, 66, 84, 61, 88, 60, 81, 79, 72, 116, 115, 86, 101, 75], "roubo_celular": [45, 45, 43, 83, 43, 52, 67, 56, 134, 101, 131, 55, 52, 27, 48, 60, 68, 50, 37, 66, 55, 54, 63, 82]}}''')
POP_PLANO_B = {"1": 1753, "5": 40449, "12": 98288, "33": 260393, "64": 478287}

try:
    base = pd.read_csv(URL_CISP, sep=";", encoding="latin1")
    pop = pd.read_csv(URL_POP, sep=";", encoding="latin1", decimal=",")
    populacao = pop.sort_values(["ano", "mes"]).groupby("circ")["pop_circ"].last()
    FONTE_ISP = "ISPDados (download agora)"
except Exception as erro:
    meses = pd.period_range("2024-10", "2026-09", freq="M")
    linhas = []
    for cisp, series in PLANO_B.items():
        for i, p in enumerate(meses):
            linhas.append({"cisp": int(cisp), "ano": p.year, "mes": p.month,
                           "fase": 2 if p >= pd.Period("2026-07", "M") else 3,
                           **{d: v[i] for d, v in series.items()}})
    base = pd.DataFrame(linhas)
    populacao = pd.Series({int(k): v for k, v in POP_PLANO_B.items()})
    FONTE_ISP = f"dados embutidos (5 CISPs, out/2024–set/2026) — download falhou: {type(erro).__name__}"

base["periodo"] = pd.to_datetime(dict(year=base["ano"], month=base["mes"], day=1)).dt.to_period("M")
if CISP not in set(base["cisp"]):
    print(f"A CISP {CISP} não está nos dados disponíveis; usando a CISP 12 como exemplo.")
    CISP = 12
serie = base[base["cisp"] == CISP].sort_values("periodo").set_index("periodo")[[DELITO, "fase"]].tail(24)
print(f"Fonte: {FONTE_ISP}")
print(f"CISP {CISP}, {DELITO}: {serie.index.min()} a {serie.index.max()} "
      f"(preliminares: {', '.join(str(p) for p in serie.index[serie['fase'] == 2]) or 'nenhum'})")

In [0]:
# Célula pronta: o dashboard da CISP. Só execute.
import matplotlib.pyplot as plt

definitivo = serie[serie["fase"] != 2]
ult = definitivo.index.max()
tri_atual = definitivo.loc[ult - 2:ult, DELITO].sum()
tri_anterior = definitivo.loc[ult - 5:ult - 3, DELITO].sum()
tri_ano_ant = serie.loc[ult - 14:ult - 12, DELITO].sum() if (ult - 14) in serie.index else np.nan
doze = serie[DELITO].tail(12).sum()
pop_cisp = float(populacao.get(CISP, np.nan))
TAXA_12M = doze / pop_cisp * 100_000 if pop_cisp else np.nan

def variacao(a, b):
    return (a - b) / b if b else np.nan

RESUMO_ISP = {
    "cisp": CISP, "delito": DELITO,
    "ultimo_trimestre_definitivo": f"{ult - 2} a {ult}", "ocorrencias_trimestre": int(tri_atual),
    "variacao_vs_trimestre_anterior": round(variacao(tri_atual, tri_anterior), 3),
    "variacao_vs_mesmo_trimestre_ano_anterior": None if np.isnan(tri_ano_ant) else round(variacao(tri_atual, tri_ano_ant), 3),
    "ocorrencias_12_meses": int(doze), "populacao_estimada": int(pop_cisp),
    "taxa_por_100_mil_12_meses": round(TAXA_12M, 1),
}

fig, ax = plt.subplots(figsize=(11, 4))
cores = ["#b23a3a" if f == 2 else "#0b2a4a" for f in serie["fase"]]
ax.bar([str(p) for p in serie.index], serie[DELITO], color=cores)
ax.set_title(f"CISP {CISP} — {DELITO} por mês (vermelho = preliminar)", loc="left")
ax.set_ylim(0, None)
ax.tick_params(axis="x", rotation=60, labelsize=8)
plt.tight_layout()
plt.show()

print(f"Último trimestre definitivo ({RESUMO_ISP['ultimo_trimestre_definitivo']}): {int(tri_atual)} ocorrências")
print(f"  contra o trimestre anterior: {variacao(tri_atual, tri_anterior):+.0%}")
if not np.isnan(tri_ano_ant):
    print(f"  contra o mesmo trimestre do ano anterior: {variacao(tri_atual, tri_ano_ant):+.0%}")
print(f"Últimos 12 meses: {int(doze)} ocorrências; {TAXA_12M:,.1f} por 100 mil habitantes")

if FONTE_ISP.startswith("ISPDados"):
    todas = base[base["periodo"] > base["periodo"].max() - 12].groupby("cisp")[DELITO].sum()
    taxas = (todas / populacao * 100_000).dropna()
    print(f"Mediana das {len(taxas)} CISPs do estado: {taxas.median():,.1f} por 100 mil "
          f"(a CISP {CISP} está acima de {(taxas < TAXA_12M).mean():.0%} delas)")

In [0]:
# Atividade 5.1 — leitura do dashboard em 30 segundos.

LEITURA_DASHBOARD = {
    "o_que": "TODO",            # indicador, delito, absoluto ou taxa
    "quando": "TODO",           # período e se há dado preliminar
    "comparado_com": "TODO",    # que base de comparação o grupo considera justa
    "e_dai": "TODO",            # que decisão este número sustenta, com o limiar do lab 1
}

## Parte 6 — Julgar notícias com `ai_decide` (15:25–15:55)

O Databricks tem uma função SQL, [`ai_decide`](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_decide), que faz **julgamentos tipados**: os mesmos tipos do System One vistos na teoria (noul, choice, score), dentro do workspace e **sem chave de API**. Para cada notícia, a pergunta é um **noul** (sim ou não, com probabilidade): *"A afirmação da notícia é sustentada pela série do ISP no mesmo recorte e período?"*. A regra de decisão fica escrita **antes**, fora do modelo:

| Probabilidade | Decisão |
| --- | --- |
| 0,7 ou mais | Sustentada: responder confirmando, com o dado |
| de 0,4 a menos de 0,7 | Faixa ambígua: revisão humana antes de qualquer nota |
| abaixo de 0,4 | Não sustentada: contestar com o dado, período e fonte |

**Como a célula roda.** Ela chama `ai_decide` para três notícias de exemplo sobre a CISP 12 e para a notícia do grupo. A função está em *Beta* e só existe em algumas regiões: se o seu workspace não tiver `ai_decide`, a célula avisa e mostra resultados de exemplo **ilustrativos** (coerentes com os dados reais do ISP). Em qualquer caso, o grupo registra **antes** o próprio julgamento, para comparar com o da máquina.

In [0]:
# Atividade 6.1 — a notícia que o grupo quer checar (Google News, sobre o recorte do grupo).

NOTICIA_DO_GRUPO = {
    "titulo": "TODO",
    "fonte_e_data": "TODO",
    "afirmacao": "TODO",           # a frase com o número que será checado
}
JULGAMENTO_HUMANO = "TODO"         # antes da IA: o grupo acha que a série sustenta? "sim", "não" ou "em parte"

In [0]:
# Célula pronta: julgamento tipado com ai_decide e regra de decisão. Só execute.
NOTICIAS_EXEMPLO = [
    {"titulo": "Roubos de rua disparam 40% na CISP 12 no segundo trimestre",
     "afirmacao": "Alta de 40% em abr–jun/2026", "p_exemplo": 0.46,
     "serie_isp": "ISP: +30% contra jan–mar/26 e +16% contra abr–jun/25; puxado por um único mês (maio)."},
    {"titulo": "Roubos de rua caem na CISP 12 no terceiro trimestre em relação a 2025",
     "afirmacao": "Queda em jul–set/2026 contra jul–set/2025", "p_exemplo": 0.74,
     "serie_isp": "ISP: 61 contra 84 (−27%), mas jul–set/26 ainda é preliminar."},
    {"titulo": "CISP 12 tem a maior taxa de roubos de rua do estado",
     "afirmacao": "Maior taxa por 100 mil do estado", "p_exemplo": 0.04,
     "serie_isp": "ISP: cerca de 352 por 100 mil em 12 meses; a CISP 5 passa de 5 mil."},
]

# A pergunta tipada: só type, instructions e criteria (o que o ai_decide aceita).
PERGUNTAS = {"evidencia_isp": {
    "type": "noul",
    "instructions": "A afirmação em noticia é sustentada pelos números em serie_isp, no mesmo recorte e período?",
    "criteria": {"true": "Os números do ISP confirmam a afirmação, no mesmo recorte e período",
                 "false": "Os números do ISP contradizem a afirmação ou não tratam do mesmo recorte e período"},
}}

def decidir(p):
    if p >= 0.7:
        return "SUSTENTADA: confirmar com o dado"
    if p >= 0.4:
        return "AMBÍGUA: revisão humana"
    return "NÃO SUSTENTADA: contestar com dado, período e fonte"

def julgar_com_ai_decide(itens):
    """itens: lista de (titulo, state). Devolve {titulo: probabilidade}. Levanta erro se ai_decide não existir."""
    linhas_sql = [(t, json.dumps(s, ensure_ascii=False)) for t, s in itens]
    spark.createDataFrame(linhas_sql, "titulo STRING, state STRING").createOrReplaceTempView("noticias_a_julgar")  # noqa: F821
    perguntas_sql = json.dumps(PERGUNTAS, ensure_ascii=False).replace("\\", "\\\\").replace("'", "\\'")
    res = spark.sql(f"""
        SELECT titulo,
               variant_get(d, '$.response.answers.evidencia_isp.probability', 'double') AS p,
               variant_get(d, '$.error_message', 'string') AS erro
        FROM (SELECT titulo, ai_decide(state, '{perguntas_sql}') AS d FROM noticias_a_julgar)
    """).toPandas()  # noqa: F821
    erros = res[res["erro"].notna()]
    if len(erros):
        raise RuntimeError(erros["erro"].iloc[0])
    return dict(zip(res["titulo"], res["p"]))

itens = [(n["titulo"], {"noticia": n["afirmacao"], "serie_isp": n["serie_isp"]}) for n in NOTICIAS_EXEMPLO]
GRUPO_PREENCHEU = all(v not in ("", "TODO") for v in NOTICIA_DO_GRUPO.values())
if GRUPO_PREENCHEU:
    itens.append((NOTICIA_DO_GRUPO["titulo"], {"noticia": NOTICIA_DO_GRUPO["afirmacao"], "serie_isp": RESUMO_ISP}))

try:
    PROBABILIDADES = julgar_com_ai_decide(itens)
    AI_DECIDE_OK = True
    print("ai_decide respondeu no seu workspace.")
except Exception as e:
    PROBABILIDADES, AI_DECIDE_OK = {}, False
    print(f"ai_decide indisponível neste workspace ({type(e).__name__}): usando resultados de exemplo ilustrativos.")

linhas = []
for n in NOTICIAS_EXEMPLO:
    p = PROBABILIDADES.get(n["titulo"], n["p_exemplo"])
    linhas.append({"Notícia": n["titulo"], "P(sustentada)": round(p, 2), "Decisão": decidir(p),
                   "Origem": "ai_decide" if AI_DECIDE_OK else "exemplo ilustrativo", "O que o ISP mostra": n["serie_isp"]})

if GRUPO_PREENCHEU:
    p = PROBABILIDADES.get(NOTICIA_DO_GRUPO["titulo"])
    linhas.append({"Notícia": NOTICIA_DO_GRUPO["titulo"], "P(sustentada)": None if p is None else round(p, 2),
                   "Decisão": decidir(p) if p is not None else "sem ai_decide: vale o julgamento humano",
                   "Origem": "ai_decide" if p is not None else "—",
                   "O que o ISP mostra": json.dumps(RESUMO_ISP, ensure_ascii=False)})
    print(f"Julgamento humano do grupo, registrado antes da IA: {JULGAMENTO_HUMANO}")

JULGAMENTOS = pd.DataFrame(linhas)
mostrar(JULGAMENTOS)

**Para discutir:** a notícia 1 tem um número real por trás (+30%), mas não o da manchete (+40%), e cai na faixa ambígua. O que a revisão humana precisa olhar? A probabilidade do `ai_decide` concorda com o julgamento humano do grupo? Em que casos o grupo aceitaria agir com probabilidade abaixo de 0,7? (Lembre do slide "o limiar nasce do custo de errar".)

---

## Parte 7 — O que levar para a aula 3 (15:55–16:20)

Na aula 3 (assíncrona) o grupo ensaia o arco inteiro sobre **uma** notícia. Escolham agora.

In [0]:
# Parte 7 — escolhas para o ensaio da aula 3.

ENSAIO_AULA_3 = {
    "noticia": "TODO",                 # título e fonte da notícia que o grupo vai checar
    "serie_isp": "TODO",               # ex. "CISP 12, roubo_rua, out/2024–set/2026"
    "base_de_comparacao": "TODO",      # ex. "mesmo trimestre do ano anterior"
    "absoluto_ou_taxa": "TODO",        # "absoluto", "taxa por 100 mil" ou "os dois"
    "ideia_de_politica_publica": "TODO",  # um rascunho: público, ação e indicador
}

# Rodada 2 da Operação Manchete (síntese das 16:30).
PORTA_RODADA_2 = "TODO"     # "A", "B", "C", "D" ou "E"
PONTOS_RODADA_2 = None      # 0 a 3, anunciado pelo professor

## Parte 8 — Checagem final (16:20–16:30)

In [0]:
# Célula pronta: confere o que falta. Só execute.

def preenchido(v):
    if v is None:
        return False
    if isinstance(v, str):
        return v.strip() not in ("", "TODO")
    if isinstance(v, dict):
        return all(preenchido(x) for x in v.values())
    if isinstance(v, list):
        return all(preenchido(x) for x in v)
    return True

itens = {
    "Parte 0 — número do grupo (1 a 8)": isinstance(GRUPO, int) and 1 <= GRUPO <= 8,
    "Parte 2 — auditoria das 10 narrativas": all(a in ("certo", "errado", "incompleto") for a in AUDITORIA),
    "Parte 3 — nome de cada grupo de texto": preenchido(NOMES_DOS_GRUPOS) and preenchido(GRUPO_MAIS_UTIL),
    "Parte 4 — decisão de anonimização": preenchido(DECISAO_ANONIMIZACAO),
    "Parte 5 — leitura do dashboard": preenchido(LEITURA_DASHBOARD),
    "Parte 6 — notícia do grupo e julgamento humano": preenchido(NOTICIA_DO_GRUPO) and preenchido(JULGAMENTO_HUMANO),
    "Parte 7 — escolhas para a aula 3": preenchido(ENSAIO_AULA_3),
    "Parte 7 — Rodada 2 registrada": PORTA_RODADA_2 in ("A", "B", "C", "D", "E") and PONTOS_RODADA_2 in (0, 1, 2, 3),
}
for item, ok in itens.items():
    print(("OK    " if ok else "FALTA ") + item)

if all(itens.values()):
    print("\nTudo pronto. Resumo para o fórum da aula 3:\n")
    print(f"Grupo {GRUPO} — CISP {CISP}, {DELITO}")
    print(f"ISP: {RESUMO_ISP['ocorrencias_12_meses']} ocorrências em 12 meses; "
          f"{RESUMO_ISP['taxa_por_100_mil_12_meses']} por 100 mil")
    print(f"Notícia para o ensaio: {ENSAIO_AULA_3['noticia']}")
    print(f"Base de comparação: {ENSAIO_AULA_3['base_de_comparacao']} | {ENSAIO_AULA_3['absoluto_ou_taxa']}")
    print(f"Anonimização: nível {DECISAO_ANONIMIZACAO['nivel_escolhido']}, compartilhar com {DECISAO_ANONIMIZACAO['pode_compartilhar_com']}")

## Material de apoio

- **Slides da aula 2**: arquitetura medalhão, modelos de linguagem, NLP em boletins, LAI e LGPD, julgamentos com probabilidade, calibração, "40% sobre quanto?", taxa por 100 mil, dashboards e política pública.
- ALVES, D. et al. Classificação de Boletins de Ocorrências através de Modelos de Linguagem baseados em BERT. LASDiGov/WCGE, SBC, 2024.
- ALMEIDA, M. G. de; BOTEGA, L. C.; LIMA, P. M. de P. Local Anonymization of Free-Text Robbery Police Reports in Brazilian Portuguese. KDMiLe, SBC, 2026 (dataset CC BY 4.0).
- [`ai_decide` — documentação do Databricks](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_decide) (Beta); o padrão de julgamentos tipados vem do [TypeSafe System One](https://docs.typesafe.ai/concepts/how-to-build-with-system-one), visto na teoria.
- [ISPDados](https://www.ispdados.rj.gov.br/) — séries por CISP e população.

**Na aula 3 (11/11, assíncrona):** leituras básicas e ensaio do arco completo sobre a notícia escolhida; teste de acesso ao ISP Microdados; Rodada 3 no fórum.